In [1]:
import emoji
import nltk
import numpy as np
import pandas as pd
import re

from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

Change the path for the csv ***HERE*** accordingly to load/save

In [2]:
IN_CSV = "../../data/train.csv"
OUT_CSV = '../../data/output_file.csv'

# TITLES = ["id", "text", "selected_text", "sentiment", "time", "age", "country", "population", "land_area", "density"]

In [3]:
df = pd.read_csv(IN_CSV, encoding="latin-1").astype(str)

In [4]:
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


Bingxi: Note for myself that twitter usernames are not case sensitive.

In [5]:
def load_emoticon_dict(dict_pth="../../data/emoji/emoticon.txt"):
    emoticon_dict = {}

    with open(dict_pth, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f]

    i = 0
    while i < len(lines):
        if lines[i] == "":
            i += 1
            continue

        emoticons = lines[i].split()
        meaning = lines[i + 1].upper()
        
        meaning = re.sub(r",", " ", meaning)
        meaning = re.sub(r"\s+", "_", meaning)
        
        for emo in emoticons:
            emo = re.sub(r"&nbsp;", " ", emo)
            emoticon_dict[emo] = meaning

        i += 3

    return emoticon_dict

In [10]:
url_pattern = r'(https?:\/\/)?(([\w\-]{1,63}\.)+[\w\-]{1,63})(\/[\w.\-]+)*([?=+%\w~\-.&]*)'
# entity_pattern = rf"@?{entity}"
user_pattern = r"@\w+"
tag_pattern = r"#\w+"
repeated_pattern = r"(.)\1\1\1+"
repeated_replace_pattern = r"\1\1\1"

In [14]:
emoticon_dict = load_emoticon_dict()
emoticons = sorted(emoticon_dict.keys(), key=len, reverse=True)
emoticon_pattern = re.compile(rf"({'|'.join(map(re.escape, emoticons))})")

In [12]:
FLAG = re.I

In [17]:
def preprocess(row):
    text = row["text"]
    
    # Concate if potential URL/Username
    # text = re.sub(r"\s+\/(\s+)?", "/", text)
    # text = re.sub(r"\@\s+", "@", text)

    # Replaces URL
    text = re.sub(url_pattern, '<URL>', text)

    # Replaces entity and other_user
    # text = re.sub(entity_pattern, "<ENTITY>", text, flags=FLAG)
    text = re.sub(user_pattern, "<USER>", text)

    # Replaces hashtags
    text = re.sub(tag_pattern, "<Tag>", text)

    # Raplaces repeated patterns
    text = re.sub(repeated_pattern, repeated_replace_pattern, text)

    text = re.sub(r"\s+", " ", text)

    text = re.sub(emoticon_pattern, lambda x: f"<EMOTICON_{emoticon_dict[x.group()]}>", text)
    
    return text

In [18]:
df["cleaned_text"] = df.apply(preprocess, axis=1)

NameError: name 'tokens' is not defined

Bingxi: 
    This is for me to test whether my code for replacing emoji is working

In [24]:
df.iloc[576]

textID                              6cb03d7c46
text                         mozart`s requiem!
selected_text                mozart`s requiem!
sentiment                              neutral
Time of Tweet                          morning
Age of User                               0-20
Country               United States of America
Population -2020                     331002651
Land Area (Km²)                      9147420.0
Density (P/Km²)                             36
cleaned_text        [mozart, `, s, requiem, !]
Name: 576, dtype: object

In [25]:
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²),cleaned_text
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60,"[I, `, d, have, responded, ,, if, I, were, going]"
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105,"[Sooo, SAD, I, will, miss, you, here, in, San,..."
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18,"[my, boss, is, bullying, me, ...]"
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164,"[what, interview, !, leave, me, alone]"
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26,"[Sons, of, *, *, *, ,, why, couldn, `, t, they..."


In [26]:
df.to_csv(OUT_CSV)